# 02a · Análisis Clínico — Univariado y Bivariado
**Fase:** Analisis estadistico de manifestaciones clinicas

1. Analisis univariado por estrato clinico
2. OR crudos con IC95% y correccion Benjamini-Hochberg
3. Heatmap de correlacion entre sintomas (colinealidad)
4. Comparativa DCSA vs DSSA

> El analisis bivariado es exploratorio. Las asociaciones independientes
> se evaluan en 02b con regresion logistica multivariada.

In [ ]:
import sys
sys.path.insert(0,'../../src')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import joblib
import warnings
from scipy.stats import chi2_contingency, fisher_exact
from statsmodels.stats.multitest import multipletests
import config
from config import (DATA_INTERIM,OUT_FIGURES,OUT_TABLES,
    COLORS,TARGET_COL,SIGNOS_ALARMA,SINTOMAS_GENERALES)
warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams.update({'figure.dpi':150,'savefig.dpi':300,
    'figure.facecolor':'white','axes.facecolor':'white',
    'axes.spines.top':False,'axes.spines.right':False,
    'axes.titlesize':13,'axes.titleweight':'bold','legend.frameon':False})

df_con = joblib.load(DATA_INTERIM/'datos_con_signos.pkl')
df_sin = joblib.load(DATA_INTERIM/'datos_sin_signos.pkl')
print(f'DCSA: {df_con.shape[0]:,} | Hosp: {df_con[TARGET_COL].mean():.1%}')
print(f'DSSA: {df_sin.shape[0]:,} | Hosp: {df_sin[TARGET_COL].mean():.1%}')

## 1. Funcion de analisis bivariado

Calcula OR crudo, IC95% (metodo Woolf) y p-valor (chi2 o Fisher).
Aplica correccion FDR Benjamini-Hochberg para comparaciones multiples.

In [ ]:
def analisis_bivariado(df, variables, target_col, etiqueta=''):
    resultados = []
    for var in variables:
        presente = df[var].map({1:1,2:0}).dropna()
        target   = df.loc[presente.index, target_col].dropna()
        idx      = presente.index.intersection(target.index)
        presente, target = presente[idx], target[idx]
        tabla = pd.crosstab(presente, target)
        if tabla.shape != (2,2):
            resultados.append({'Variable':var,'OR':np.nan,'IC95_inf':np.nan,
                'IC95_sup':np.nan,'p_valor':np.nan})
            continue
        a,b = max(tabla.iloc[1,1],0.5), max(tabla.iloc[1,0],0.5)
        c,d = max(tabla.iloc[0,1],0.5), max(tabla.iloc[0,0],0.5)
        OR  = (a*d)/(b*c)
        se  = np.sqrt(1/a+1/b+1/c+1/d)
        lo  = np.exp(np.log(OR)-1.96*se)
        hi  = np.exp(np.log(OR)+1.96*se)
        _,p,_,_ = chi2_contingency([[a,b],[c,d]])
        pct_h  = tabla.iloc[1,1]/tabla.iloc[:,1].sum()*100
        pct_nh = tabla.iloc[1,0]/tabla.iloc[:,0].sum()*100
        resultados.append({'Variable':var,
            'Hosp_%':round(pct_h,1),'NoHosp_%':round(pct_nh,1),
            'OR':round(OR,3),'IC95_inf':round(lo,3),'IC95_sup':round(hi,3),
            'p_valor':p})
    res = pd.DataFrame(resultados)
    _, p_adj,_,_ = multipletests(res['p_valor'].fillna(1), method='fdr_bh')
    res['p_adj_BH']    = p_adj.round(4)
    res['Significativo'] = res['p_adj_BH'] < 0.05
    print(f'Bivariado {etiqueta}: {len(df):,} registros')
    return res.sort_values('OR', ascending=False)
print('Funcion lista')

In [ ]:
# Analisis DCSA
todas = SINTOMAS_GENERALES + SIGNOS_ALARMA
tabla_con = analisis_bivariado(df_con, todas, TARGET_COL, 'DCSA')
display(tabla_con)
tabla_con.to_csv(OUT_TABLES/'bivariado_DCSA.csv', index=False)

In [ ]:
# Forest plot OR crudo — DCSA
df_fp = tabla_con.dropna(subset=['OR']).sort_values('OR')
fig, ax = plt.subplots(figsize=(10,7))
for i, (_, f) in enumerate(df_fp.iterrows()):
    color  = COLORS['con_signos'] if f['Variable'] in SIGNOS_ALARMA else COLORS['sin_signos']
    marker = 'D' if f['Significativo'] else 'o'
    ax.errorbar(f['OR'], i,
        xerr=[[f['OR']-f['IC95_inf']],[f['IC95_sup']-f['OR']]],
        fmt=marker, capsize=4, markersize=8, color=color,
        elinewidth=1.5, markeredgecolor='white', markeredgewidth=0.8)
    ax.text(f['IC95_sup']+0.08, i,
        f"{f['OR']:.2f} [{f['IC95_inf']:.2f}-{f['IC95_sup']:.2f}]",
        va='center', fontsize=8)
ax.set_yticks(range(len(df_fp)))
ax.set_yticklabels(df_fp['Variable'], fontsize=9)
ax.axvline(1, color=COLORS['neutral'], linestyle='--', linewidth=1.2)
ax.set_xlabel('Odds Ratio crudo (IC 95%)')
ax.set_title('Forest plot OR crudo — Dengue con signos de alarma')
leyenda = [mpatches.Patch(color=COLORS['con_signos'],label='Signo de alarma'),
    mpatches.Patch(color=COLORS['sin_signos'],label='Sintoma general'),
    plt.Line2D([0],[0],marker='D',color='gray',markersize=8,linewidth=0,label='p_adj<0.05'),
    plt.Line2D([0],[0],marker='o',color='gray',markersize=7,linewidth=0,label='p_adj>=0.05')]
ax.legend(handles=leyenda, loc='lower right', fontsize=9)
ax.set_xlim(0, df_fp['IC95_sup'].max()+1)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
fig.savefig(OUT_FIGURES/'04_forest_OR_crudo_DCSA.png',
    dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# Heatmap correlacion phi entre sintomas
# Correlacion phi = Pearson para variables binarias
# Alta correlacion (>0.5) indica informacion redundante para el modelo
df_bin = df_con[todas].replace({1:1, 2:0})
corr   = df_bin.corr(method='pearson')

fig, ax = plt.subplots(figsize=(12,10))
import numpy as np
mask = np.eye(len(corr), dtype=bool)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-0.5, vmax=0.5,
    linewidths=0.5, linecolor='white',
    annot_kws={'size':8}, square=True, ax=ax)
n_gen = len(SINTOMAS_GENERALES)
ax.axhline(n_gen, color='black', linewidth=2)
ax.axvline(n_gen, color='black', linewidth=2)
ax.set_title('Correlacion phi entre manifestaciones clinicas del dengue')
plt.tight_layout()
fig.savefig(OUT_FIGURES/'05_heatmap_correlacion.png',
    dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# Analisis DSSA
tabla_sin = analisis_bivariado(df_sin, todas, TARGET_COL, 'DSSA')
display(tabla_sin)
tabla_sin.to_csv(OUT_TABLES/'bivariado_DSSA.csv', index=False)

## Resumen

**DCSA:** Caida plaquetas, dolor abdominal y hemorragia mucosas = mayores OR crudos.

**DSSA:** Asociaciones mas debiles; en este estrato la decision de hospitalizar
responde mas a factores no registrados (acceso, comorbilidades).

**Siguiente:** `02b_regresion_logistica_clinica.ipynb`